In [2]:
!unzip Dataset-Movies.zip

Archive:  Dataset-Movies.zip
  inflating: dataset_train.csv       
  inflating: __MACOSX/._dataset_train.csv  
  inflating: dataset_test.csv        
  inflating: __MACOSX/._dataset_test.csv  


In [14]:
import random
import zipfile

import numpy as np
import pandas as pd

%pip install torch
import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    precision_recall_fscore_support,
    f1_score,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

train_df = pd.read_csv("dataset_train.csv")
test_df = pd.read_csv("dataset_test.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)

display(train_df.head())
display(test_df.head())

train_df["genre_list"] = train_df["genre"].apply(
    lambda x: [g.strip() for g in x.split(",")]
)

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(train_df["genre_list"])

genres = list(mlb.classes_)

print("Géneros:")
print(genres)
print("Y shape:", Y.shape)

train_df["text"] = (
    "Title: " + train_df["movie_name"].fillna("") +
    ". Plot: " + train_df["description"].fillna("")
)

test_df["text"] = (
    "Title: " + test_df["movie_name"].fillna("") +
    ". Plot: " + test_df["description"].fillna("")
)

display(train_df[["movie_name", "genre", "text"]].head())

Train: (8475, 3)
Test: (942, 2)


,movie_name,genre,description
0,Silent Hill,"Horror, Mystery","Rose, a desperate mother takes her adopted dau..."
1,Breaking the Waves,"Drama, Romance","In a small and conservative Scottish village, ..."
2,Wind Chill,"Drama, Horror, Thriller",Two college students share a ride home for the...
3,Godmothered,"Family, Fantasy, Comedy",A young and unskilled fairy godmother that ven...
4,Donkey Skin,"Fantasy, Comedy, Music, Romance",A fairy godmother helps a princess disguise he...


,movie_name,description
0,Opposites Attract,"She's a divorce lawyer, single mother and perp..."
1,A Turtle's Tale: Sammy's Adventures,A sea turtle who was hatched in 1959 spends th...
2,My Stepmother Is an Alien,Trying to rescue her home planet from destruct...
3,You've Got Mail,"Book superstore magnate, Joe Fox and independe..."
4,The Thing,"In the winter of 1982, a twelve-man research t..."


Géneros:
['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']
Y shape: (8475, 18)


,movie_name,genre,text
0,Silent Hill,"Horror, Mystery","Title: Silent Hill. Plot: Rose, a desperate mo..."
1,Breaking the Waves,"Drama, Romance",Title: Breaking the Waves. Plot: In a small an...
2,Wind Chill,"Drama, Horror, Thriller",Title: Wind Chill. Plot: Two college students ...
3,Godmothered,"Family, Fantasy, Comedy",Title: Godmothered. Plot: A young and unskille...
4,Donkey Skin,"Fantasy, Comedy, Music, Romance",Title: Donkey Skin. Plot: A fairy godmother he...


In [5]:
print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Y shape:", Y.shape)

Train: (8475, 5)
Test: (942, 3)
Y shape: (8475, 18)


In [6]:
X = train_df["text"].tolist()
X_test = test_df["text"].tolist()

X_train, X_dev, y_train, y_dev = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

print("X_train:", len(X_train))
print("X_dev:", len(X_dev))
print("y_train:", y_train.shape)
print("y_dev:", y_dev.shape)



X_train: 6780
X_dev: 1695
y_train: (6780, 18)
y_dev: (1695, 18)


In [7]:
def compute_metrics_official(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    hl = hamming_loss(y_true, y_pred)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl
    }


def compute_metrics_debug(y_true, y_pred):
    metrics = compute_metrics_official(y_true, y_pred)
    metrics["avg_labels_pred"] = y_pred.sum(axis=1).mean()
    metrics["avg_labels_true"] = y_true.sum(axis=1).mean()
    return metrics

In [8]:
MODEL_NAME = "roberta-base"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [9]:
class MovieGenreDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

        return item

In [10]:
train_dataset = MovieGenreDataset(X_train, y_train, tokenizer, MAX_LEN)
dev_dataset = MovieGenreDataset(X_dev, y_dev, tokenizer, MAX_LEN)
test_dataset = MovieGenreDataset(X_test, np.zeros((len(X_test), len(genres))), tokenizer, MAX_LEN)

print(len(train_dataset), len(dev_dataset), len(test_dataset))

6780 1695 942


In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(genres),
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return compute_metrics_official(labels, preds)

In [13]:
%pip install "transformers[torch]"
training_args = TrainingArguments(
    output_dir="./roberta_base_results",
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics_trainer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [16]:
trainer.train()

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.295952,0.251375,0.139233,0.389837,0.565725,0.326648,0.104261,20.411800,83.040000,5.193000
2,0.220488,0.223983,0.194690,0.507706,0.651340,0.449643,0.092691,21.301900,79.570000,4.976000
3,0.187094,0.217069,0.204130,0.559230,0.649291,0.504236,0.089249,20.495600,82.701000,5.172000
4,0.165911,0.215209,0.218289,0.561719,0.646635,0.506856,0.087807,20.582700,82.351000,5.150000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3392, training_loss=0.2173611398013133, metrics={'train_runtime': 1339.4217, 'train_samples_per_second': 20.248, 'train_steps_per_second': 2.532, 'total_flos': 3568298450042880.0, 'train_loss': 0.2173611398013133, 'epoch': 4.0})

In [17]:
print("MPS disponible:", torch.backends.mps.is_available())
print("MPS construido:", torch.backends.mps.is_built())

MPS disponible: True
MPS construido: True


In [18]:
eval_results = trainer.evaluate()
display(eval_results)

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
0.165911,0.215209,4,0.218289,0.561719,0.646635,0.506856,0.087807,20.497600,82.693000,5.171000


{'eval_loss': 0.21520932018756866,
 'eval_accuracy': 0.2182890855457227,
 'eval_f1': 0.5617187196635475,
 'eval_precision': 0.6466345505148152,
 'eval_recall': 0.5068562728401951,
 'eval_hamming_loss': 0.08780727630285153,
 'eval_runtime': 20.4976,
 'eval_samples_per_second': 82.693,
 'eval_steps_per_second': 5.171}

In [19]:
dev_outputs = trainer.predict(dev_dataset)

dev_logits = dev_outputs.predictions
dev_probs = 1 / (1 + np.exp(-dev_logits))

print(dev_probs.shape)

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


(1695, 18)


In [20]:
def find_best_thresholds(y_true, y_scores, genres, thresholds=np.arange(0.05, 0.96, 0.01)):
    best_thresholds = []
    best_f1s = []

    for j, genre in enumerate(genres):
        best_t = 0.5
        best_f1 = 0.0

        for t in thresholds:
            y_pred_j = (y_scores[:, j] >= t).astype(int)
            f1 = f1_score(y_true[:, j], y_pred_j, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresholds.append(best_t)
        best_f1s.append(best_f1)

    return np.array(best_thresholds), np.array(best_f1s)

In [21]:
best_thresholds_roberta, best_f1s_roberta = find_best_thresholds(
    y_dev,
    dev_probs,
    genres
)

thresholds_roberta_df = pd.DataFrame({
    "genre": genres,
    "threshold": best_thresholds_roberta,
    "dev_f1_class": best_f1s_roberta,
    "support_dev": y_dev.sum(axis=0)
}).sort_values("genre")

display(thresholds_roberta_df)

,genre,threshold,dev_f1_class,support_dev
0,Action,0.50,0.759142,411
1,Adventure,0.26,0.649776,278
2,Animation,0.29,0.685879,172
3,Comedy,0.28,0.760991,625
4,Crime,0.26,0.666667,273
5,Drama,0.40,0.752927,789
6,Family,0.24,0.702128,178
7,Fantasy,0.29,0.582090,189
8,History,0.27,0.559633,90
9,Horror,0.28,0.707113,231


In [22]:
y_pred_roberta_thr = (dev_probs >= best_thresholds_roberta).astype(int)

metrics_roberta_thr = compute_metrics_debug(y_dev, y_pred_roberta_thr)
display(metrics_roberta_thr)

print(classification_report(
    y_dev,
    y_pred_roberta_thr,
    target_names=genres,
    zero_division=0
))

{'accuracy': 0.20766961651917404,
 'f1': 0.6426985429933788,
 'precision': 0.6188440504750434,
 'recall': 0.6769163853430451,
 'hamming_loss': 0.09180596525729269,
 'avg_labels_pred': np.float64(2.9415929203539823),
 'avg_labels_true': np.float64(2.6601769911504425)}

                 precision    recall  f1-score   support

         Action       0.79      0.73      0.76       411
      Adventure       0.55      0.78      0.65       278
      Animation       0.68      0.69      0.69       172
         Comedy       0.76      0.76      0.76       625
          Crime       0.59      0.76      0.67       273
          Drama       0.70      0.81      0.75       789
         Family       0.67      0.74      0.70       178
        Fantasy       0.55      0.62      0.58       189
        History       0.48      0.68      0.56        90
         Horror       0.68      0.73      0.71       231
          Music       0.60      0.58      0.59        53
        Mystery       0.51      0.52      0.51       163
        Romance       0.71      0.68      0.70       268
Science Fiction       0.85      0.77      0.80       213
       TV Movie       0.11      0.24      0.15        17
       Thriller       0.68      0.80      0.74       471
            War       0.76    

In [23]:
print(trainer.args.device)

mps


In [24]:
test_outputs = trainer.predict(test_dataset)

test_logits = test_outputs.predictions
test_probs_roberta = 1 / (1 + np.exp(-test_logits))

print(test_probs_roberta.shape)

test_pred_bin_roberta = (test_probs_roberta >= best_thresholds_roberta).astype(int)

# Asegurar al menos un género por película
for i in range(test_pred_bin_roberta.shape[0]):
    if test_pred_bin_roberta[i].sum() == 0:
        best_class = test_probs_roberta[i].argmax()
        test_pred_bin_roberta[i, best_class] = 1

test_pred_labels_roberta = mlb.inverse_transform(test_pred_bin_roberta)

test_pred_genres_roberta = [
    ", ".join(labels)
    for labels in test_pred_labels_roberta
]

submission_roberta_df = pd.DataFrame({
    "movie_name": test_df["movie_name"],
    "description": test_df["description"],
    "genre": test_pred_genres_roberta
})

display(submission_roberta_df.head())
print(submission_roberta_df.shape)

submission_roberta_df.to_csv("dataset_test_preds.csv", index=False)

with zipfile.ZipFile("ILN05-RoBERTaBase-PerClassTH.zip", "w") as zipf:
    zipf.write("dataset_test_preds.csv")

with zipfile.ZipFile("ILN05-RoBERTaBase-PerClassTH.zip", "r") as zipf:
    print(zipf.namelist())

check_df = pd.read_csv("dataset_test_preds.csv")

print(check_df.shape)
print(check_df.columns.tolist())
print(check_df["genre"].isna().sum())
print((check_df["genre"].str.len() == 0).sum())
display(check_df.head())

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


(942, 18)


,movie_name,description,genre
0,Opposites Attract,"She's a divorce lawyer, single mother and perp...","Comedy, Drama, Romance"
1,A Turtle's Tale: Sammy's Adventures,A sea turtle who was hatched in 1959 spends th...,"Adventure, Animation, Comedy, Family"
2,My Stepmother Is an Alien,Trying to rescue her home planet from destruct...,"Comedy, Drama, Romance, Science Fiction"
3,You've Got Mail,"Book superstore magnate, Joe Fox and independe...","Comedy, Drama, Romance"
4,The Thing,"In the winter of 1982, a twelve-man research t...","Horror, Science Fiction, Thriller"


(942, 3)
['dataset_test_preds.csv']
(942, 3)
['movie_name', 'description', 'genre']
0
0


,movie_name,description,genre
0,Opposites Attract,"She's a divorce lawyer, single mother and perp...","Comedy, Drama, Romance"
1,A Turtle's Tale: Sammy's Adventures,A sea turtle who was hatched in 1959 spends th...,"Adventure, Animation, Comedy, Family"
2,My Stepmother Is an Alien,Trying to rescue her home planet from destruct...,"Comedy, Drama, Romance, Science Fiction"
3,You've Got Mail,"Book superstore magnate, Joe Fox and independe...","Comedy, Drama, Romance"
4,The Thing,"In the winter of 1982, a twelve-man research t...","Horror, Science Fiction, Thriller"
